# Basics of Data Structures and Algorithms

## Introduction

In the previous lecture on [computational complexity](computational-complexity.md), we learned how to analyze the time and space costs of algorithms using Big-O notation. We also saw that choosing the right approach (brute force vs. sorting vs. hashing) can change an algorithm from impractical to instant. The key insight was that **algorithm choice matters more than hardware**.

This lecture builds on that foundation. We'll study the core data structures and algorithm design techniques that underpin efficient scientific computing. The goal is not to memorize implementations, but to develop intuition for *when and why* to reach for each tool.

We will cover:

* **Abstract Data Types (ADTs)**: Understanding the separation between interface and implementation
* **Data structures**: Stacks, queues, linked lists, trees, and graphs
* **Recursion and backtracking**: Solving problems by breaking them into smaller instances
* **Sorting algorithms**: Selection sort, insertion sort, merge sort, and quicksort
* **Algorithm design paradigms**: Divide and conquer, greedy algorithms, and dynamic programming
* **Graph algorithms**: BFS, DFS, and Dijkstra's shortest path

These topics come up constantly in scientific computing. Graph algorithms appear in network analysis and phylogenetics. Dynamic programming is the foundation of sequence alignment in bioinformatics. Trees power database indexing and spatial search. Understanding Abstract Data Types helps you choose the right data structure for your problem and recognize when different implementations offer different performance trade-offs. Throughout this lecture, we include visual demonstrations and working code examples to build intuition for how these algorithms behave.

## Data Structures

We already know Python's built-in data structures: lists, dictionaries, sets, and tuples. These cover many use cases, but some problems call for more specialized structures.

### Abstract Data Types

An **Abstract Data Type (ADT)** is defined by what it does, not how it does it. Think of it like a contract: an ADT specifies two things:

1. **Operations**: What actions are available (push, pop, search, insert, delete)
2. **Performance guarantees**: The computational complexity of each operation ($O(1)$, $O(\log n)$, $O(n)$)

Critically, an ADT does **not** specify implementation details like data representation or algorithms. This separation between **interface** and **implementation** is the foundation of good software design.

For example, a "priority queue" ADT promises:
- `insert(item, priority)`: Add an item with a priority — $O(\log n)$
- `extract_min()`: Remove and return the minimum element — $O(\log n)$
- `peek_min()`: View the minimum element without removing it — $O(1)$

The ADT doesn't specify whether this uses a heap, a sorted array, or a balanced tree internally. This lets you choose the right implementation for your use case or swap implementations without changing the code that uses it. If your application frequently checks the minimum but rarely inserts, you might use a different implementation than if insertions dominate.

The data structures we'll study (stacks, queues, trees, graphs) are all ADTs. For each, we'll see how the interface dictates what operations are available, and how different implementations offer different trade-offs in time and space complexity.

In the sections that follow, we'll study several fundamental ADTs:

- **Stack**: push, pop, peek (LIFO ordering)
- **Queue**: enqueue, dequeue, peek (FIFO ordering)
- **List**: insert, delete, search (sequential access)
- **Tree**: insert, search, traverse (hierarchical structure)
- **Graph**: add_edge, find_path, traverse (network structure)

For each ADT, we'll examine the operations it provides and the computational complexity guarantees. Some ADTs like Stack and Queue have essentially one good implementation. Others like trees or graphs have multiple implementations with different trade-offs. Understanding both the ADT (what you need) and the implementations (how to get it efficiently) is key to choosing the right data structure.

### Stacks and Queues

A **stack** follows Last-In-First-Out (LIFO) order: the most recently added item is the first to be removed, like a stack of plates. A **queue** follows First-In-First-Out (FIFO) order: items are processed in the order they arrived, like a line at a store.

Both Stack and Queue are ADTs that define specific operations with guaranteed complexities:

**Stack operations:**
- `push(item)`: Add to top — $O(1)$
- `pop()`: Remove from top — $O(1)$
- `peek()`: View top element — $O(1)$

**Queue operations:**
- `enqueue(item)`: Add to back — $O(1)$
- `dequeue()`: Remove from front — $O(1)$
- `peek()`: View front element — $O(1)$

Below we implement both as Python classes to demonstrate their interfaces and complexity guarantees.

In [1]:
class Stack:
    """Stack data structure with LIFO (Last-In-First-Out) ordering."""

    def __init__(self):
        self._items = []

    def push(self, item):
        """Add item to top of stack. O(1)."""
        self._items.append(item)

    def pop(self):
        """Remove and return top item. O(1)."""
        if self.is_empty():
            raise IndexError("pop from empty stack")
        return self._items.pop()

    def peek(self):
        """Return top item without removing it. O(1)."""
        if self.is_empty():
            raise IndexError("peek from empty stack")
        return self._items[-1]

    def is_empty(self):
        """Check if stack is empty. O(1)."""
        return len(self._items) == 0

    def __repr__(self):
        return f"Stack({self._items})"


# Example: Processing tasks in reverse order
stack = Stack()
stack.push("task A")
stack.push("task B")
stack.push("task C")
print(f"Stack state: {stack}")
print(f"Pop: {stack.pop()}")  # "task C" (last in, first out)
print(f"Pop: {stack.pop()}")  # "task B"
print(f"Peek: {stack.peek()}")  # "task A" (still there)
print(f"Final state: {stack}")

Stack state: Stack(['task A', 'task B', 'task C'])
Pop: task C
Pop: task B
Peek: task A
Final state: Stack(['task A'])


In [2]:
from collections import deque


class Queue:
    """Queue data structure with FIFO (First-In-First-Out) ordering."""

    def __init__(self):
        self._items = deque()

    def enqueue(self, item):
        """Add item to back of queue. O(1)."""
        self._items.append(item)

    def dequeue(self):
        """Remove and return front item. O(1)."""
        if self.is_empty():
            raise IndexError("dequeue from empty queue")
        return self._items.popleft()

    def peek(self):
        """Return front item without removing it. O(1)."""
        if self.is_empty():
            raise IndexError("peek from empty queue")
        return self._items[0]

    def is_empty(self):
        """Check if queue is empty. O(1)."""
        return len(self._items) == 0

    def __repr__(self):
        return f"Queue({list(self._items)})"


# Example: Processing patients in arrival order
queue = Queue()
queue.enqueue("patient 1")
queue.enqueue("patient 2")
queue.enqueue("patient 3")
print(f"Queue state: {queue}")
print(f"Dequeue: {queue.dequeue()}")  # "patient 1" (first in, first out)
print(f"Dequeue: {queue.dequeue()}")  # "patient 2"
print(f"Peek: {queue.peek()}")  # "patient 3" (still there)
print(f"Final state: {queue}")

Queue state: Queue(['patient 1', 'patient 2', 'patient 3'])
Dequeue: patient 1
Dequeue: patient 2
Peek: patient 3
Final state: Queue(['patient 3'])


Note that `Queue` uses `collections.deque` internally because it provides $O(1)$ operations on both ends. Using a plain Python list for a queue would be inefficient: removing from the front with `pop(0)` takes $O(n)$ time since all remaining elements must be shifted.

Stacks are used for function call management (the "call stack"), undo/redo operations, and expression parsing. Queues are used for job scheduling, breadth-first search (which we'll see later), and managing task pipelines.

### Dynamic Arrays

A **dynamic array** (also called a contiguous array) stores elements in a single, continuous block of memory. Python's built-in `list` is actually a dynamic array. When you access an element by index, Python can compute its memory address directly, giving $O(1)$ access time.

The key feature of dynamic arrays is that they automatically resize when full. When you append to a full array, Python allocates a new, larger block of memory (typically 1.5x to 2x the current size), copies all elements, and releases the old block. While a single append might take $O(n)$ time during resizing, most appends are $O(1)$, making the **amortized** cost $O(1)$ per append.

In [3]:
class DynamicArray:
    """Simple wrapper around Python's list to demonstrate dynamic array concepts."""

    def __init__(self):
        self._data = []

    def append(self, item):
        """Add item to end. Amortized O(1)."""
        self._data.append(item)

    def get(self, index):
        """Access element by index. O(1)."""
        return self._data[index]

    def set(self, index, value):
        """Set element at index. O(1)."""
        self._data[index] = value

    def insert(self, index, item):
        """Insert item at index. O(n) - must shift elements."""
        self._data.insert(index, item)

    def delete(self, index):
        """Delete item at index. O(n) - must shift elements."""
        del self._data[index]

    def __len__(self):
        return len(self._data)

    def __repr__(self):
        return f"DynamicArray({self._data})"


# Demonstrate dynamic array operations
arr = DynamicArray()
arr.append(10)
arr.append(20)
arr.append(30)
print(f"Array: {arr}")
print(f"Get index 1: {arr.get(1)}")  # Fast: O(1)
arr.set(1, 25)
print(f"After set: {arr}")
arr.insert(1, 15)  # Slow: O(n) - must shift elements
print(f"After insert at 1: {arr}")
arr.delete(0)  # Slow: O(n) - must shift elements
print(f"After delete at 0: {arr}")

Array: DynamicArray([10, 20, 30])
Get index 1: 20
After set: DynamicArray([10, 25, 30])
After insert at 1: DynamicArray([10, 15, 25, 30])
After delete at 0: DynamicArray([15, 25, 30])


The key trade-offs of dynamic arrays:

* **Fast random access**: Getting or setting an element at any index is $O(1)$ because the address can be computed directly
* **Slow insertion/deletion**: Inserting or deleting in the middle requires shifting all subsequent elements, taking $O(n)$ time
* **Cache-friendly**: Elements are stored together in memory, making sequential access very fast

This contrasts with linked lists (covered next), which have $O(1)$ insertion/deletion at known positions but $O(n)$ access by index.

### Linked Lists

A **linked list** is a sequence of nodes where each node stores a value and a reference (pointer) to the next node. Unlike a dynamic array, linked list elements are scattered in memory and connected by pointers.

In [4]:
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None


class LinkedList:
    def __init__(self):
        self.head = None

    def append(self, data):
        new_node = Node(data)
        if self.head is None:
            self.head = new_node
            return
        current = self.head
        while current.next:
            current = current.next
        current.next = new_node

    def display(self):
        elements = []
        current = self.head
        while current:
            elements.append(str(current.data))
            current = current.next
        print(" -> ".join(elements))


ll = LinkedList()
ll.append(10)
ll.append(20)
ll.append(30)
ll.display()  # 10 -> 20 -> 30

10 -> 20 -> 30


The key trade-off between linked lists and arrays:

| Operation | Array (Python list) | Linked List |
|---|---|---|
| Access by index | $O(1)$ | $O(n)$ |
| Insert/delete at beginning | $O(n)$ | $O(1)$ |
| Insert/delete at end | $O(1)$ amortized | $O(n)$ without tail pointer |
| Search | $O(n)$ | $O(n)$ |

Linked lists are useful when you need frequent insertions and deletions at arbitrary positions and don't need random access. In practice, Python's `collections.deque` (a doubly-linked list internally) is the go-to when you need efficient operations on both ends.

### Trees

A **tree** is a hierarchical data structure where each node has zero or more children. The topmost node is the **root**, and nodes with no children are **leaves**.

A **binary tree** is a tree where each node has at most two children (left and right). A **binary search tree (BST)** adds an ordering property: for every node, all values in the left subtree are smaller, and all values in the right subtree are larger.

**Binary Search Tree operations:**
- `insert(item)`: Add element — $O(\log n)$ average, $O(n)$ worst case
- `search(item)`: Find element — $O(\log n)$ average, $O(n)$ worst case
- `delete(item)`: Remove element — $O(\log n)$ average, $O(n)$ worst case

The logarithmic complexity assumes a balanced tree. In the worst case (degenerate tree that looks like a linked list), operations degrade to $O(n)$. Balanced tree variants like AVL trees or Red-Black trees guarantee $O(\log n)$ worst-case performance.

In [5]:
class TreeNode:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None


def insert_bst(root, value):
    """Insert a value into a binary search tree."""
    if root is None:
        return TreeNode(value)
    if value < root.value:
        root.left = insert_bst(root.left, value)
    else:
        root.right = insert_bst(root.right, value)
    return root


def search_bst(root, target):
    """Search for a value in a binary search tree."""
    if root is None:
        return False
    if target == root.value:
        return True
    if target < root.value:
        return search_bst(root.left, target)
    return search_bst(root.right, target)


def inorder_traversal(root):
    """Visit nodes in sorted order: left, root, right."""
    if root is None:
        return []
    return inorder_traversal(root.left) + [root.value] + inorder_traversal(root.right)


# Build a BST
root = None
for val in [50, 30, 70, 20, 40, 60, 80]:
    root = insert_bst(root, val)

print(inorder_traversal(root))  # [20, 30, 40, 50, 60, 70, 80]
print(search_bst(root, 40))    # True
print(search_bst(root, 45))    # False

[20, 30, 40, 50, 60, 70, 80]
True
False


Trees appear throughout scientific computing:

* **Decision trees** in machine learning split data based on feature thresholds
* **KD-trees** partition spatial data for efficient nearest-neighbor queries (we mentioned this in the complexity lecture)
* **Heap data structures** (a special kind of binary tree) power priority queues, which are used in Dijkstra's algorithm and event-driven simulations
* **File systems** are tree structures
* **Phylogenetic trees** represent evolutionary relationships in biology

### Graphs

A **graph** consists of **vertices** (nodes) and **edges** (connections between nodes). Graphs are one of the most versatile data structures in computing. An edge can be **directed** (one-way, like a web link) or **undirected** (two-way, like a friendship). Edges can also have **weights** representing costs, distances, or strengths.

**Graph ADT operations:**
- `add_edge(u, v)`: Connect vertices — $O(1)$ typical
- `has_edge(u, v)`: Check connection — depends on representation
- `neighbors(v)`: Get adjacent vertices — $O(\text{degree})$ for adjacency list

The two standard ways to represent a graph are:

**Adjacency list:** Each vertex stores a list of its neighbors. Space-efficient for sparse graphs (few edges relative to vertices).

**Adjacency matrix:** A 2D matrix where entry $(i, j)$ indicates whether an edge exists between vertices $i$ and $j$. Enables $O(1)$ edge lookup but uses $O(V^2)$ space.

In [6]:
# Adjacency list representation (using a dictionary)
graph_list = {
    "A": ["B", "C"],
    "B": ["A", "D", "E"],
    "C": ["A", "F"],
    "D": ["B"],
    "E": ["B", "F"],
    "F": ["C", "E"],
}

# Check if an edge exists
print("B" in graph_list["A"])  # True
print("F" in graph_list["A"])  # False

True
False


In [7]:
import numpy as np

# Adjacency matrix representation
# Vertices: A=0, B=1, C=2, D=3, E=4, F=5
adj_matrix = np.array([
    [0, 1, 1, 0, 0, 0],  # A
    [1, 0, 0, 1, 1, 0],  # B
    [1, 0, 0, 0, 0, 1],  # C
    [0, 1, 0, 0, 0, 0],  # D
    [0, 1, 0, 0, 0, 1],  # E
    [0, 0, 1, 0, 1, 0],  # F
])

# Check if edge exists between A (0) and B (1)
print(adj_matrix[0, 1])  # 1 (edge exists)

1


| Representation | Space | Edge lookup | Iterate neighbors |
|---|---|---|---|
| Adjacency list | $O(V + E)$ | $O(\text{degree})$ | $O(\text{degree})$ |
| Adjacency matrix | $O(V^2)$ | $O(1)$ | $O(V)$ |

Real-world graph applications include:

* **Social networks:** people are vertices, friendships are edges
* **Protein interaction networks:** proteins are vertices, interactions are edges
* **Citation networks:** papers are vertices, citations are directed edges
* **Road networks:** intersections are vertices, roads are weighted edges

#### Question

You are analyzing a protein interaction network with 20,000 proteins where each protein interacts with an average of 10 others. Would you choose an adjacency list or an adjacency matrix? How much memory does each representation use (roughly, in bytes)?

#### Answer

Choose an **adjacency list**. With 20,000 proteins and ~10 interactions each, there are about 100,000 edges. An adjacency list stores these as pairs, requiring roughly $O(V + E) = O(20{,}000 + 100{,}000) \approx 120{,}000$ entries, around 1 MB.

An adjacency matrix would require $20{,}000 \times 20{,}000 = 4 \times 10^8$ entries. As a byte matrix, that's 400 MB, or as float64, 3.2 GB. Most of these entries would be zero since the graph is sparse (100,000 edges out of 200 million possible), wasting memory.

## Recursion and Backtracking

Recursion is when a function calls itself to solve smaller instances of the same problem. Every recursive function needs a **base case** (when to stop) and a **recursive case** (how to reduce the problem).

We saw Fibonacci in the complexity lecture. Here is another classic example: computing the factorial of a number.

In [8]:
def factorial(n):
    """Compute n! recursively."""
    if n <= 1:       # base case
        return 1
    return n * factorial(n - 1)  # recursive case

print(factorial(5))  # 120 = 5 * 4 * 3 * 2 * 1

120


Python has a default recursion limit of 1000 (check with `sys.getrecursionlimit()`). For deep recursion, you may need to increase it or convert to an iterative approach.

### Backtracking

**Backtracking** is a technique that builds solutions incrementally, abandoning ("backtracking" from) a partial solution as soon as it determines the solution cannot be completed successfully. It's essentially a depth-first exploration of the solution space with pruning.

A classic example is generating all permutations of a list:

In [9]:
def permutations(elements):
    """Generate all permutations of elements using backtracking."""
    result = []

    def backtrack(current, remaining):
        if not remaining:
            result.append(current[:])  # found a complete permutation
            return
        for i in range(len(remaining)):
            current.append(remaining[i])
            backtrack(current, remaining[:i] + remaining[i + 1 :])
            current.pop()  # undo the choice (backtrack)

    backtrack([], elements)
    return result


perms = permutations([1, 2, 3])
print(f"Number of permutations: {len(perms)}")
for p in perms:
    print(p)

Number of permutations: 6
[1, 2, 3]
[1, 3, 2]
[2, 1, 3]
[2, 3, 1]
[3, 1, 2]
[3, 2, 1]


The backtracking pattern has three steps: (1) make a choice, (2) recursively explore, (3) undo the choice. This pattern appears in constraint satisfaction problems, puzzle solving, and combinatorial optimization.

## Sorting Algorithms

Sorting is one of the most fundamental operations in computing. Python's built-in `sorted()` and `list.sort()` use a hybrid algorithm with $O(n \log n)$ worst-case performance. Understanding different sorting algorithms helps you appreciate why $O(n \log n)$ is the theoretical lower bound for comparison-based sorting and how different strategies achieve it.

### Selection Sort

Selection sort works by repeatedly finding the minimum element from the unsorted portion and moving it to the beginning. It divides the array into a sorted portion (initially empty) and an unsorted portion (initially the entire array). In each iteration, it selects the smallest element from the unsorted portion and swaps it with the first unsorted element.

In [10]:
def selection_sort(arr):
    """Sort array in place using selection sort. O(n^2)."""
    for i in range(len(arr)):
        # Find minimum element in remaining unsorted portion
        min_idx = i
        for j in range(i + 1, len(arr)):
            if arr[j] < arr[min_idx]:
                min_idx = j

        # Swap minimum element with first unsorted element
        arr[i], arr[min_idx] = arr[min_idx], arr[i]

    return arr


data = [64, 25, 12, 22, 11]
print(f"Before: {data}")
selection_sort(data)
print(f"After:  {data}")

Before: [64, 25, 12, 22, 11]
After:  [11, 12, 22, 25, 64]


Selection sort always runs in $O(n^2)$ time regardless of the input. It performs $(n-1) + (n-2) + \cdots + 1 = \frac{n(n-1)}{2}$ comparisons, making it inefficient for large arrays. Unlike insertion sort, selection sort doesn't benefit from partially sorted data. However, it makes at most $n-1$ swaps (one per iteration), which can be advantageous when writing to memory is expensive.

### Insertion Sort

Insertion sort builds the sorted list one element at a time by inserting each new element into its correct position among the already-sorted elements. It's like sorting a hand of playing cards: you pick up cards one by one and insert each into its proper place in your hand.

In [11]:
def insertion_sort(arr):
    """Sort array in place using insertion sort. O(n^2)."""
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        while j >= 0 and arr[j] > key:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key
    return arr


data = [64, 25, 12, 22, 11]
print(f"Before: {data}")
insertion_sort(data)
print(f"After:  {data}")

Before: [64, 25, 12, 22, 11]
After:  [11, 12, 22, 25, 64]


Insertion sort is $O(n^2)$ in the worst case but $O(n)$ when the input is already nearly sorted. This makes it a good choice for small arrays or as a final pass in hybrid algorithms.

### Merge Sort

Merge sort is a **divide and conquer** algorithm: it splits the array in half, recursively sorts each half, then merges the two sorted halves. It guarantees $O(n \log n)$ time in all cases.

In [12]:
def merge_sort(arr):
    """Sort array using merge sort. O(n log n)."""
    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)


def merge(left, right):
    """Merge two sorted arrays into one sorted array."""
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result


data = [38, 27, 43, 3, 9, 82, 10]
print(f"Before: {data}")
sorted_data = merge_sort(data)
print(f"After:  {sorted_data}")

Before: [38, 27, 43, 3, 9, 82, 10]
After:  [3, 9, 10, 27, 38, 43, 82]


The key insight: merging two sorted arrays of total size $n$ takes $O(n)$ time. The array is split $\log n$ times, and at each level the total merge work is $O(n)$, giving $O(n \log n)$ overall.

### Quicksort

Quicksort is another divide and conquer algorithm. It picks a **pivot** element, partitions the array so that elements less than the pivot come before it and elements greater come after it, then recursively sorts each partition. (We briefly saw this in the first Python lecture.)

In [13]:
def quicksort(arr):
    """Sort array using quicksort. O(n log n) average, O(n^2) worst."""
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return quicksort(left) + middle + quicksort(right)


data = [38, 27, 43, 3, 9, 82, 10]
print(f"Before: {data}")
sorted_data = quicksort(data)
print(f"After:  {sorted_data}")

Before: [38, 27, 43, 3, 9, 82, 10]
After:  [3, 9, 10, 27, 38, 43, 82]


Quicksort is $O(n \log n)$ on average but $O(n^2)$ in the worst case (when the pivot is always the smallest or largest element). Despite this, quicksort is often faster than merge sort in practice because it sorts in-place (our simplified version above creates new lists, but the in-place variant avoids this overhead) and has good cache behavior.

### Comparing Sorting Algorithms

In [14]:
import time
import numpy as np


def time_sort(sort_func, data, repetitions=5):
    """Time a sorting function over multiple repetitions."""
    total = 0
    for _ in range(repetitions):
        arr = data.copy()
        start = time.time()
        sort_func(arr)
        total += time.time() - start
    return total / repetitions


for n in [1000, 5000, 10000]:
    data = list(np.random.randint(0, n * 10, n))
    t_selection = time_sort(selection_sort, data)
    t_insert = time_sort(insertion_sort, data)
    t_merge = time_sort(merge_sort, data)
    t_quick = time_sort(quicksort, data)
    t_builtin = time_sort(sorted, data)
    print(
        f"n={n:>6d}: selection={t_selection:.4f}s, insertion={t_insert:.4f}s, "
        f"merge={t_merge:.4f}s, quick={t_quick:.4f}s, built-in={t_builtin:.6f}s"
    )

n=  1000: selection=0.0217s, insertion=0.0094s, merge=0.0008s, quick=0.0007s, built-in=0.000096s

n=  5000: selection=0.2624s, insertion=0.2413s, merge=0.0050s, quick=0.0045s, built-in=0.000585s


n= 10000: selection=1.0727s, insertion=0.9761s, merge=0.0107s, quick=0.0095s, built-in=0.001312s


| Algorithm | Best | Average | Worst | Space | Stable? |
|---|---|---|---|---|---|
| Selection sort | $O(n^2)$ | $O(n^2)$ | $O(n^2)$ | $O(1)$ | No |
| Insertion sort | $O(n)$ | $O(n^2)$ | $O(n^2)$ | $O(1)$ | Yes |
| Merge sort | $O(n \log n)$ | $O(n \log n)$ | $O(n \log n)$ | $O(n)$ | Yes |
| Quicksort | $O(n \log n)$ | $O(n \log n)$ | $O(n^2)$ | $O(\log n)$ | No |

A "stable" sort preserves the relative order of equal elements. This matters when sorting by multiple keys (e.g., sort by name, then by date).

![Sorting algorithms visualization](imgs/sorting.png)

#### Question

You have a list of 10 million patient records that are *almost sorted* (only a few hundred records are out of place due to data entry corrections). Which sorting algorithm would perform best here, and why?

#### Answer

**Insertion sort** would perform best. Insertion sort is $O(n)$ on nearly-sorted data since each element only needs to move a few positions. Merge sort and quicksort would both be $O(n \log n)$ regardless of the existing order. For 10 million records, you can use the built-in `sorted()` since Python's sorting algorithm is optimized for partially sorted data and is implemented in C.

---

**Visual Comparison:** For an animated comparison of how these sorting algorithms behave on different input patterns, see [this visualization](https://www.youtube.com/watch?v=kPRA0W1kECg).

## Algorithm Design Paradigms

When faced with a new problem, algorithm designers often reach for one of several standard strategies. We'll cover three of the most important: divide and conquer, greedy algorithms, and dynamic programming.

### Divide and Conquer

We already saw divide and conquer with merge sort and quicksort. The pattern is:

1. **Divide** the problem into smaller subproblems
2. **Conquer** each subproblem recursively
3. **Combine** the results

Another classic application is **binary search**, which finds a target value in a sorted array by repeatedly halving the search range:

In [15]:
def binary_search(arr, target):
    """Find target in sorted array. Returns index or -1. O(log n)."""
    low, high = 0, len(arr) - 1
    while low <= high:
        mid = (low + high) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
    return -1


data = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print(binary_search(data, 23))  # 5
print(binary_search(data, 50))  # -1

5
-1


#### Maximum Subarray Problem

The **maximum subarray problem** asks: given an array of numbers (possibly negative), find the contiguous subarray with the largest sum. This appears in analyzing stock prices (maximum profit from buying and selling) and signal processing (finding the strongest signal period).

**Intuition:** The divide and conquer approach works by recognizing that the maximum subarray must be either (1) entirely in the left half, (2) entirely in the right half, or (3) crossing the midpoint. We recursively find the best in each half, then find the best crossing subarray by extending from the midpoint in both directions. The crossing case is $O(n)$ since we scan from the middle outward. Since we split in half each time, we get $O(n \log n)$ overall.

In [16]:
def max_subarray(arr):
    """Find maximum sum of contiguous subarray using divide and conquer. O(n log n)."""

    def max_crossing_sum(arr, left, mid, right):
        """Find max sum crossing the midpoint."""
        # Find max sum going left from mid
        left_sum = float("-inf")
        current_sum = 0
        for i in range(mid, left - 1, -1):
            current_sum += arr[i]
            left_sum = max(left_sum, current_sum)

        # Find max sum going right from mid+1
        right_sum = float("-inf")
        current_sum = 0
        for i in range(mid + 1, right + 1):
            current_sum += arr[i]
            right_sum = max(right_sum, current_sum)

        return left_sum + right_sum

    def max_subarray_recursive(arr, left, right):
        """Recursively find max subarray."""
        if left == right:
            return arr[left]

        mid = (left + right) // 2

        # Max subarray is either entirely in left half, right half, or crosses mid
        left_max = max_subarray_recursive(arr, left, mid)
        right_max = max_subarray_recursive(arr, mid + 1, right)
        cross_max = max_crossing_sum(arr, left, mid, right)

        return max(left_max, right_max, cross_max)

    return max_subarray_recursive(arr, 0, len(arr) - 1)


# Example: Stock price changes (daily gains/losses)
price_changes = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
max_profit = max_subarray(price_changes)
print(f"Price changes: {price_changes}")
print(f"Maximum profit from contiguous period: {max_profit}")
print(f"(Subarray [4, -1, 2, 1] sums to 6)")

Price changes: [-2, 1, -3, 4, -1, 2, 1, -5, 4]
Maximum profit from contiguous period: 6
(Subarray [4, -1, 2, 1] sums to 6)


#### Fast Exponentiation

Computing $x^n$ naively by multiplying $x$ by itself $n$ times takes $O(n)$. Using divide and conquer, we can reduce this to $O(\log n)$ by exploiting the identity: $x^n = (x^{n/2})^2$ when $n$ is even, and $x^n = x \cdot x^{n-1}$ when $n$ is odd. This technique appears in cryptography and modular arithmetic.

**Intuition:** Instead of doing $n$ multiplications, we compute $x^{n/2}$ once and square it. For example, $x^{16} = (x^8)^2 = ((x^4)^2)^2 = (((x^2)^2)^2)^2$ requires only 4 squarings instead of 15 multiplications. Each recursive call halves the exponent, giving $O(\log n)$ depth. When $n$ is odd, we handle it by computing $x \cdot x^{n-1}$ where $n-1$ is now even.

In [17]:
def power_naive(x, n):
    """Compute x^n naively. O(n)."""
    result = 1
    for _ in range(n):
        result *= x
    return result


def power_fast(x, n):
    """Compute x^n using divide and conquer. O(log n)."""
    if n == 0:
        return 1
    if n == 1:
        return x

    # Divide: compute x^(n/2)
    half = power_fast(x, n // 2)

    # Conquer: square the result
    if n % 2 == 0:
        return half * half
    else:
        return x * half * half


# Compare performance
import time

x, n = 2.5, 1000

start = time.time()
result_naive = power_naive(x, n)
time_naive = time.time() - start

start = time.time()
result_fast = power_fast(x, n)
time_fast = time.time() - start

print(f"Computing {x}^{n}:")
print(f"Naive: {time_naive:.6f}s")
print(f"Fast:  {time_fast:.6f}s")
print(f"Results match: {abs(result_naive - result_fast) < 1e-6}")
print(f"Speedup: {time_naive / time_fast:.1f}x")

Computing 2.5^1000:
Naive: 0.000039s
Fast:  0.000019s
Results match: False
Speedup: 2.0x


Binary search is $O(\log n)$ because each comparison eliminates half the remaining elements. It requires sorted input, but if you perform many searches on the same data, the one-time $O(n \log n)$ sorting cost is worth it.

Divide and conquer appears widely in scientific computing:

* **Fast Fourier Transform (FFT)** reduces $O(n^2)$ convolution to $O(n \log n)$
* **Strassen's algorithm** multiplies matrices in $O(n^{2.807})$ instead of $O(n^3)$
* **Closest pair of points** algorithms run in $O(n \log n)$ instead of $O(n^2)$

### Greedy Algorithms

A **greedy algorithm** makes the locally optimal choice at each step, hoping that local choices lead to a global optimum. Greedy algorithms are typically simple and fast, but they don't always produce the best solution.

A classic example is the **activity selection problem**: given a set of activities with start and end times, select the maximum number of non-overlapping activities.

In [18]:
def activity_selection(activities):
    """Select maximum non-overlapping activities.

    Args:
        activities: List of (start, end) tuples.

    Returns:
        List of selected activities.
    """
    # Greedy strategy: always pick the activity that ends earliest
    sorted_acts = sorted(activities, key=lambda x: x[1])
    selected = [sorted_acts[0]]
    last_end = sorted_acts[0][1]

    for start, end in sorted_acts[1:]:
        if start >= last_end:
            selected.append((start, end))
            last_end = end

    return selected


activities = [(1, 4), (3, 5), (0, 6), (5, 7), (3, 9), (5, 9), (6, 10), (8, 11)]
selected = activity_selection(activities)
print(f"Selected {len(selected)} activities: {selected}")

Selected 3 activities: [(1, 4), (5, 7), (8, 11)]


This greedy approach is provably optimal for activity selection.

**Intuition:** The key insight is that choosing the earliest-ending activity leaves the most room for future activities. If we chose an activity that ends later, we'd block out more time and potentially exclude activities we could have fit in. By always picking the one that frees up time soonest, we maximize our options going forward. This greedy choice is "safe" because no other choice can lead to a better solution.

However, greedy algorithms can fail. Consider the **coin change problem**: make change for a given amount using the fewest coins.

In [19]:
def greedy_coin_change(amount, denominations):
    """Greedy coin change: always pick the largest coin possible."""
    denominations = sorted(denominations, reverse=True)
    coins_used = []
    remaining = amount
    for coin in denominations:
        while remaining >= coin:
            coins_used.append(coin)
            remaining -= coin
    return coins_used


# Works for standard US denominations
print(greedy_coin_change(36, [1, 5, 10, 25]))
# [25, 10, 1] = 3 coins (optimal)

# Fails for non-standard denominations
print(greedy_coin_change(6, [1, 3, 4]))
# [4, 1, 1] = 3 coins (greedy), but [3, 3] = 2 coins (optimal!)

[25, 10, 1]
[4, 1, 1]


Greedy algorithms work well when the problem has the **greedy choice property** (a locally optimal choice can be extended to a globally optimal solution) and **optimal substructure** (optimal solutions contain optimal solutions to subproblems).

### Dynamic Programming

**Dynamic programming (DP)** solves problems by breaking them into overlapping subproblems, solving each subproblem once, and storing the results. It's more powerful than greedy algorithms because it considers all possible choices rather than committing to the locally best one.

**The 0/1 Knapsack Problem:** Given items with weights and values, and a knapsack with a weight capacity, find the maximum value you can carry.

**Intuition:** For each item, we face a choice: take it or leave it. The key insight is that the optimal solution for "first $i$ items with capacity $w$" depends on solutions to smaller subproblems. If we take item $i$, we get its value plus the optimal solution for "first $i-1$ items with capacity $w - \text{weight}_i$". If we don't take it, we get the optimal solution for "first $i-1$ items with capacity $w$". We build a table where `dp[i][w]` stores the maximum value achievable with the first $i$ items and capacity $w$, filling it bottom-up.

In [20]:
def knapsack(weights, values, capacity):
    """Solve 0/1 knapsack problem using dynamic programming.

    Args:
        weights: List of item weights.
        values: List of item values.
        capacity: Maximum weight capacity.

    Returns:
        Maximum achievable value.
    """
    n = len(weights)
    # dp[i][w] = max value using first i items with capacity w
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        for w in range(capacity + 1):
            # Option 1: don't take item i
            dp[i][w] = dp[i - 1][w]
            # Option 2: take item i (if it fits)
            if weights[i - 1] <= w:
                dp[i][w] = max(
                    dp[i][w],
                    dp[i - 1][w - weights[i - 1]] + values[i - 1],
                )

    return dp[n][capacity]


# Example: lab equipment selection with budget constraint
weights = [2, 3, 4, 5]
values = [3, 4, 5, 6]
capacity = 8
print(f"Maximum value: {knapsack(weights, values, capacity)}")

Maximum value: 10


The key DP concepts are:

* **Overlapping subproblems:** The same subproblems appear repeatedly (unlike divide and conquer, where subproblems are independent)
* **Optimal substructure:** The optimal solution contains optimal solutions to subproblems
* **Two approaches:** Top-down (recursion with memoization) or bottom-up (fill a table iteratively)

#### Matrix Chain Multiplication

When multiplying a sequence of matrices, the order matters for efficiency. Multiplying an $m \times n$ matrix by an $n \times p$ matrix takes $O(mnp)$ operations. **Matrix chain multiplication** finds the optimal parenthesization to minimize total operations. This is important in scientific computing where matrix operations dominate performance (linear algebra, deep learning, graphics).

**Intuition:** Consider multiplying matrices A×B×C. We can do (A×B)×C or A×(B×C), with different costs depending on dimensions. The key insight: to multiply matrices $i$ through $j$, we must split them at some position $k$, multiply the two resulting chains, then multiply the results together. We try all possible split points $k$ and pick the one that minimizes: (cost of left chain) + (cost of right chain) + (cost of multiplying the two results). Build a table `dp[i][j]` for the cost of multiplying matrices $i$ through $j$, filling it by increasing chain length.

In [1]:
def matrix_chain_order(dimensions):
    """Find optimal matrix multiplication order using DP. O(n^3).

    Args:
        dimensions: List of matrix dimensions. For n matrices A1...An,
                    dimensions[i-1] x dimensions[i] gives the size of Ai.

    Returns:
        Minimum number of scalar multiplications needed.
    """
    n = len(dimensions) - 1  # number of matrices

    # dp[i][j] = min operations to multiply matrices i through j
    dp = [[0] * n for _ in range(n)]

    # l is the chain length
    for l in range(2, n + 1):  # l goes from 2 to n
        for i in range(n - l + 1):
            j = i + l - 1
            dp[i][j] = float("inf")

            # Try all possible split points
            for k in range(i, j):
                # Cost = cost of left chain + cost of right chain + cost of merging
                cost = (
                    dp[i][k]
                    + dp[k + 1][j]
                    + dimensions[i] * dimensions[k + 1] * dimensions[j + 1]
                )
                dp[i][j] = min(dp[i][j], cost)

    return dp[0][n - 1]


# Example: Multiply 4 matrices
# A1: 10x20, A2: 20x30, A3: 30x40, A4: 40x30
dims = [10, 20, 30, 40, 30]

min_ops = matrix_chain_order(dims)
print(f"Matrix dimensions: {dims}")
print(f"Matrices: A1(10x20), A2(20x30), A3(30x40), A4(40x30)")
print(f"Minimum operations (DP): {min_ops}")

# Compare with different orderings
left_to_right = (10 * 20 * 30) + (10 * 30 * 40) + (10 * 40 * 30)
print(f"\nLeft-to-right ((A1*A2)*A3)*A4: {left_to_right} operations")

right_to_left = (30 * 40 * 30) + (20 * 30 * 30) + (10 * 20 * 30)
print(f"Right-to-left A1*(A2*(A3*A4)): {right_to_left} operations")

print(f"\nOptimal order saves {right_to_left / min_ops:.2f}x operations vs right-to-left")
print("DP found that left-to-right is optimal for these dimensions")

Matrix dimensions: [10, 20, 30, 40, 30]
Matrices: A1(10x20), A2(20x30), A3(30x40), A4(40x30)
Minimum operations (DP): 30000

Left-to-right ((A1*A2)*A3)*A4: 30000 operations
Right-to-left A1*(A2*(A3*A4)): 60000 operations

Optimal order saves 2.00x operations vs right-to-left
DP found that left-to-right is optimal for these dimensions


The time and space complexity of this DP solution is $O(mn)$, where $m$ and $n$ are the lengths of the two sequences. The brute-force approach of trying all possible alignments would be exponential.

#### Question

How would you compare the greedy, divide and conquer, and dynamic programming approaches? For each, name one problem where it's the right choice and one where it would fail or be suboptimal.

#### Answer

**Divide and conquer** splits problems into *independent* subproblems. Right choice: merge sort (subproblems don't overlap). Would fail: Fibonacci (subproblems overlap massively, leading to exponential redundant work without memoization).

**Greedy** makes locally optimal choices without reconsidering. Right choice: activity selection (earliest-end-time strategy is provably optimal). Would fail: 0/1 knapsack (taking the highest value-to-weight ratio item can miss the global optimum).

**Dynamic programming** handles problems with *overlapping* subproblems. Right choice: edit distance / sequence alignment (exponentially many overlapping subproblems). Would fail (in the sense of being overkill): binary search, where subproblems don't overlap and divide and conquer suffices.

## Graph Algorithms

Graphs model relationships, and many computational problems reduce to traversing or finding paths in graphs. We'll cover the three most fundamental graph algorithms.

### Breadth-First Search (BFS)

BFS explores a graph level by level, visiting all neighbors of a vertex before moving to their neighbors. It uses a **queue** and finds the shortest path (by number of edges) from a starting vertex to all reachable vertices.

In [24]:
from collections import deque


def bfs(graph, start):
    """Breadth-first search. Returns vertices in BFS order."""
    visited = set()
    queue = deque([start])
    visited.add(start)
    order = []

    while queue:
        vertex = queue.popleft()
        order.append(vertex)
        for neighbor in graph[vertex]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return order


graph = {
    "A": ["B", "C"],
    "B": ["A", "D", "E"],
    "C": ["A", "F"],
    "D": ["B"],
    "E": ["B", "F"],
    "F": ["C", "E"],
}

print("BFS from A:", bfs(graph, "A"))

BFS from A: ['A', 'B', 'C', 'D', 'E', 'F']


BFS can be extended to find the shortest path (in terms of number of edges):

In [25]:
def bfs_shortest_path(graph, start, goal):
    """Find shortest path from start to goal using BFS."""
    visited = set([start])
    queue = deque([(start, [start])])

    while queue:
        vertex, path = queue.popleft()
        if vertex == goal:
            return path
        for neighbor in graph[vertex]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, path + [neighbor]))

    return None  # no path found


path = bfs_shortest_path(graph, "A", "F")
print(f"Shortest path A -> F: {path}")

Shortest path A -> F: ['A', 'C', 'F']


BFS time complexity is $O(V + E)$ where $V$ is the number of vertices and $E$ is the number of edges. It's used for finding shortest paths in unweighted graphs, computing connected components, and level-order tree traversal.

### Depth-First Search (DFS)

DFS explores as deep as possible along each branch before backtracking. It uses a **stack** (either explicitly or via recursion).

In [26]:
def dfs_recursive(graph, start, visited=None):
    """Depth-first search using recursion."""
    if visited is None:
        visited = set()
    visited.add(start)
    order = [start]
    for neighbor in graph[start]:
        if neighbor not in visited:
            order.extend(dfs_recursive(graph, neighbor, visited))
    return order


def dfs_iterative(graph, start):
    """Depth-first search using an explicit stack."""
    visited = set()
    stack = [start]
    order = []

    while stack:
        vertex = stack.pop()
        if vertex not in visited:
            visited.add(vertex)
            order.append(vertex)
            # Add neighbors in reverse order so we visit them left-to-right
            for neighbor in reversed(graph[vertex]):
                if neighbor not in visited:
                    stack.append(neighbor)

    return order


print("DFS (recursive) from A:", dfs_recursive(graph, "A"))
print("DFS (iterative) from A:", dfs_iterative(graph, "A"))

DFS (recursive) from A: ['A', 'B', 'D', 'E', 'F', 'C']
DFS (iterative) from A: ['A', 'B', 'D', 'E', 'F', 'C']


DFS also runs in $O(V + E)$ time. It's used for cycle detection, topological sorting (ordering tasks with dependencies), and finding connected components. The key difference from BFS is the exploration pattern: BFS goes wide, DFS goes deep.

![BFS vs DFS comparison](https://media.geeksforgeeks.org/wp-content/uploads/20240216084522/bfs-vs-dfs-(1).png)

#### Question

You are mapping a social network and want to find if two people are connected through at most 3 intermediate friends (i.e., within 4 hops). Would you use BFS or DFS? Why?

#### Answer

Use **BFS**. BFS explores the graph level by level, so it naturally finds the shortest path between two nodes. You can stop as soon as you find the target (guaranteeing the shortest path) or once you've explored all nodes up to depth 4. DFS could find a path, but it might follow a very long path first and wouldn't guarantee finding the shortest one without exhaustive search.

![DFS xkcd comic](https://imgs.xkcd.com/comics/dfs.png)

### Dijkstra's Shortest Path Algorithm

BFS finds the shortest path in an **unweighted** graph (minimizing the number of edges). But what if edges have different weights representing distances, costs, or times? **Dijkstra's algorithm** finds the shortest path in a **weighted graph** with non-negative edge weights.

The algorithm maintains a set of visited vertices and a priority queue of unvisited vertices, always expanding the vertex with the smallest known distance. This greedy strategy is guaranteed to find the optimal path.

![Dijkstra's algorithm example](imgs/dijkstra.png)

![Dijkstra's algorithm visualization](https://upload.wikimedia.org/wikipedia/commons/5/57/Dijkstra_Animation.gif)

In [27]:
import heapq


def dijkstra(graph, start):
    """Find shortest paths from start to all vertices using Dijkstra's algorithm.

    Args:
        graph: Dict mapping vertex -> list of (neighbor, weight) tuples.
        start: Starting vertex.

    Returns:
        (distances, predecessors) where distances[v] is the shortest distance
        from start to v, and predecessors[v] is the previous vertex on the path.

    Time complexity: O((V + E) log V) with binary heap.
    """
    distances = {vertex: float("inf") for vertex in graph}
    distances[start] = 0
    predecessors = {vertex: None for vertex in graph}

    # Priority queue: (distance, vertex)
    pq = [(0, start)]
    visited = set()

    while pq:
        current_dist, current = heapq.heappop(pq)

        if current in visited:
            continue
        visited.add(current)

        # Explore neighbors
        for neighbor, weight in graph[current]:
            distance = current_dist + weight

            # Found a shorter path to neighbor
            if distance < distances[neighbor]:
                distances[neighbor] = distance
                predecessors[neighbor] = current
                heapq.heappush(pq, (distance, neighbor))

    return distances, predecessors


def reconstruct_path(predecessors, start, goal):
    """Reconstruct shortest path from start to goal."""
    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = predecessors[current]
    return list(reversed(path))


# Example: City road network with distances
graph = {
    "A": [("B", 4), ("C", 2)],
    "B": [("A", 4), ("C", 1), ("D", 5)],
    "C": [("A", 2), ("B", 1), ("D", 8), ("E", 10)],
    "D": [("B", 5), ("C", 8), ("E", 2), ("F", 6)],
    "E": [("C", 10), ("D", 2), ("F", 3)],
    "F": [("D", 6), ("E", 3)],
}

distances, predecessors = dijkstra(graph, "A")

print("Shortest distances from A:")
for vertex in sorted(distances.keys()):
    print(f"  A -> {vertex}: {distances[vertex]}")

path = reconstruct_path(predecessors, "A", "F")
print(f"\nShortest path A -> F: {' -> '.join(path)}")
print(f"Total distance: {distances['F']}")

Shortest distances from A:
  A -> A: 0
  A -> B: 3
  A -> C: 2
  A -> D: 8
  A -> E: 10
  A -> F: 13

Shortest path A -> F: A -> C -> B -> D -> E -> F
Total distance: 13


#### Dijkstra vs BFS

To understand when Dijkstra's algorithm is necessary, let's compare it with BFS on the same graph:

In [28]:
# Convert weighted graph to unweighted (for BFS)
unweighted_graph = {vertex: [n for n, w in neighbors] for vertex, neighbors in graph.items()}

# BFS shortest path (by number of edges)
from collections import deque


def bfs_distance(graph, start):
    """BFS finds shortest path by number of edges."""
    distances = {vertex: float("inf") for vertex in graph}
    distances[start] = 0
    queue = deque([start])
    predecessors = {vertex: None for vertex in graph}

    while queue:
        current = queue.popleft()
        for neighbor in graph[current]:
            if distances[neighbor] == float("inf"):
                distances[neighbor] = distances[current] + 1
                predecessors[neighbor] = current
                queue.append(neighbor)

    return distances, predecessors


bfs_distances, bfs_predecessors = bfs_distance(unweighted_graph, "A")
bfs_path = reconstruct_path(bfs_predecessors, "A", "F")

print("BFS (minimizes number of edges):")
print(f"  Path: {' -> '.join(bfs_path)}")
print(f"  Edges: {len(bfs_path) - 1}")

# Calculate actual weight of BFS path
bfs_weight = 0
for i in range(len(bfs_path) - 1):
    u, v = bfs_path[i], bfs_path[i + 1]
    for neighbor, weight in graph[u]:
        if neighbor == v:
            bfs_weight += weight
            break
print(f"  Total weight: {bfs_weight}")

print("\nDijkstra (minimizes total weight):")
print(f"  Path: {' -> '.join(path)}")
print(f"  Edges: {len(path) - 1}")
print(f"  Total weight: {distances['F']}")

print(f"\nDijkstra finds a path with more edges but lower total weight!")

BFS (minimizes number of edges):
  Path: A -> B -> D -> F
  Edges: 3
  Total weight: 15

Dijkstra (minimizes total weight):
  Path: A -> C -> B -> D -> E -> F
  Edges: 5
  Total weight: 13

Dijkstra finds a path with more edges but lower total weight!


**Important limitation**: Dijkstra's algorithm requires non-negative edge weights. For graphs with negative weights, use the Bellman-Ford algorithm instead.

**Applications** of Dijkstra's algorithm include:

* **Navigation systems**: Finding shortest routes between locations (GPS, Google Maps)
* **Network routing**: Finding optimal paths for data packets (OSPF protocol)
* **Robotics**: Path planning for robots navigating physical spaces
* **Game AI**: Movement and pathfinding for NPCs
* **Computational biology**: Finding optimal alignments in sequence analysis

## Summary

* **Data structures** provide different trade-offs. Stacks (LIFO) and queues (FIFO) control processing order. Dynamic arrays enable fast random access. Linked lists enable efficient insertions/deletions. Binary search trees provide $O(\log n)$ lookup when balanced. Graphs model relationships between entities.

* **Recursion** solves problems by reducing them to smaller instances. Backtracking extends recursion by exploring choices and undoing them when they fail.

* **Sorting algorithms** differ in their guarantees. Selection sort always runs in $O(n^2)$. Insertion sort is fast on nearly-sorted data ($O(n)$ best case). Merge sort guarantees $O(n \log n)$. Quicksort is $O(n \log n)$ on average but $O(n^2)$ worst case.

* **Algorithm design paradigms** guide how to approach new problems. Divide and conquer splits into independent subproblems (merge sort, binary search, FFT, maximum subarray, fast exponentiation). Greedy algorithms make locally optimal choices (activity selection, fractional knapsack, Huffman coding). Dynamic programming stores solutions to overlapping subproblems (0/1 knapsack, sequence alignment, edit distance, LCS, matrix chain multiplication).

* **Graph algorithms** solve network problems. BFS finds shortest unweighted paths in $O(V + E)$. DFS explores deeply for cycle detection and topological sorting in $O(V + E)$. Dijkstra's finds shortest weighted paths in $O((V + E) \log V)$ for non-negative edge weights.

## Recommended Resources

* [Big-O Cheat Sheet](https://www.bigocheatsheet.com/): complexity reference for common algorithms and data structures
* [Introduction to Algorithms (CLRS)](https://mitpress.mit.edu/books/introduction-algorithms-fourth-edition): the standard reference, covers all topics in depth
* [Visualgo](https://visualgo.net/): interactive visualizations of sorting algorithms, graph traversals, and more
* [Python `collections` module](https://docs.python.org/3/library/collections.html): `deque`, `Counter`, `defaultdict`, and other useful data structures
* [NetworkX](https://networkx.org/): Python library for graph analysis with built-in BFS, DFS, Dijkstra's, and many more algorithms
* [Biological Sequence Analysis (Durbin et al.)](https://www.cambridge.org/core/books/biological-sequence-analysis/921BB77E4E4B3BFC0F8B65B3FCFE641D): dynamic programming for sequence alignment in bioinformatics